# Classwork: The Analyst's Toolkit

**Names:** _(pair)_

After learning about different tools (or a toolkit-- Descriptive statistics, Correlation, Regression/Multiple regression, Chi-square test) of working with data in the previous Mods, in this acitivity you will apply your real skills on the data you have.  
  
  Suppose your company hands you `customers.csv` (40 customers: `age`, `income`, `spending`, `region`, `subscribed`) and a list of questions. You are the analyst. Answer each one with the right tool/technique you decide to apply.

Work in pairs. Write answers in the markdown cells provided.

## Setup
Import the libraries you'll need and load the data.

In [1]:
# import pandas (and numpy, scipy.stats, statsmodels.api as you need them)
# load customers.csv into a DataFrame
# take a first look (head, info)

import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm

df = pd.read_csv("./customers.csv")

df.head()
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 40 entries, 0 to 39
Data columns (total 6 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  40 non-null     int64  
 1   age          40 non-null     int64  
 2   income       38 non-null     float64
 3   spending     40 non-null     float64
 4   region       40 non-null     str    
 5   subscribed   40 non-null     str    
dtypes: float64(2), int64(2), str(2)
memory usage: 2.0 KB


## Part 0: Know your data, then clean it

This data is dirty on purpose. Find and fix **three** problems before you analyze anything.

In [2]:
# HINT 1: a numeric column has missing values -> decide how to fill them (mean? median? why?)
# HINT 2: one category label is written in the wrong case -> standardize it
# HINT 3: one value is impossible for its column -> treat it as missing, then fill
# tip: check .isna().sum() and the unique values of the category column


df["income"] = df["income"].fillna(df["income"].median())

df["region"] = df["region"].replace("east", "East")

df.loc[df["income"] < 0, "income"] = df["income"].median()

print(df.isna().sum())
print(df["region"].unique())
print(df["income"].min())


customer_id    0
age            0
income         0
spending       0
region         0
subscribed     0
dtype: int64
<StringArray>
['West', 'South', 'East']
Length: 3, dtype: str
53.3


**Explain each issue:** what you found and how you handled it.

Problems: 
Income is missing 2 values, regions has East and east as 2 septerate things, and someones income is negative.

Fixes: 
If any incomes are null: set it to the median income.
Since I knew the thing wrong with regions i just changed that specific thing and set "east" to "East".
I changed any income that was under 0 to be set to the median income.


## Part 1: Pick the tool for each question

For each question: (a) name the data types involved, (b) name the tool, (c) run it, (d) answer in one plain sentence.

### Q1. Do customers who earn more also spend more?

In [3]:
# data types: income = ?, spending = ?
# tool: ?
# hint: two continuous variables moving together

correlation, p_value = stats.pearsonr(df["income"], df["spending"])

print("Correlation:", correlation)
print("p-value:", p_value)



Correlation: 0.7172222861355748
p-value: 1.9185294399175956e-07


*Your answer:*
data types: Income = continuous, Spending = continuous
tool: Correlation

yes customers who earn more do also spend more.


### Q2. Is a customer's region related to whether they subscribed?

In [4]:
# data types: region = ?, subscribed = ?
# tool: ?
# hint: two categorical variables -> build a table with pd.crosstab, then test it

table = pd.crosstab(df["region"], df["subscribed"])
print(table)

chi2, p, dof, expected = stats.chi2_contingency(table)

print("Chi-square:", chi2)
print("p-value:", p)
print("Expected counts:")
print(expected)


subscribed  No  Yes
region             
East         5   16
South        6    3
West         9    1
Chi-square: 13.161904761904761
p-value: 0.0013865281628130974
Expected counts:
[[10.5 10.5]
 [ 4.5  4.5]
 [ 5.   5. ]]


*Your answer:*
data types: Region = categorial, Subscribed = categorial
tool: Chi square test

yes the region is related to whether or not a customer is subscribed. (p-value is less than 0.05)

### Q3. Can we predict spending from income and age together? How much do they explain? (You can review your Assignment 4 for this)

In [5]:
# data types: predicting a continuous outcome from TWO predictors
# tool: ?
# hint: remember to add a constant; look at R-squared and each predictor's p-value

X = df[["income", "age"]]
X = sm.add_constant(X)

y = df["spending"]

model = sm.OLS(y, X).fit()

print(model.summary())


                            OLS Regression Results                            
Dep. Variable:               spending   R-squared:                       0.520
Model:                            OLS   Adj. R-squared:                  0.494
Method:                 Least Squares   F-statistic:                     20.04
Date:                Tue, 06 Oct 2026   Prob (F-statistic):           1.27e-06
Time:                        22:59:57   Log-Likelihood:                -131.11
No. Observations:                  40   AIC:                             268.2
Df Residuals:                      37   BIC:                             273.3
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         19.2622      6.000      3.210      0.0

*Your answer:*
data types: Spending = continuous, income and age = continuous
tool: multiple regression

yes. income and age explain 52% of the variation in spending 

### Q4. Which region has the most customers, and what is the average income overall?

In [6]:
# hint: count a categorical column
# take the mean of a numeric column
print(df["region"].value_counts())
print(df["income"].mean())



region
East     21
West     10
South     9
Name: count, dtype: int64
86.46


*Your answer:*
the East has the most customers.
the average income overall is 86.46

## Part 2: The reveal

Answer these two in a sentence each (no code needed).

**a)** Look back at your four tools (Correlation, Chi-square test, Multiple regression, & Descriptive statistic)). What decided which tool you used each time? (Hint: it was not the wording of the question.)

*Your answer:*
The type of varialbes and what I wanted to find.


**b)** In Q1 you found a relationship. Can you tell the company that higher income *causes* higher spending? Why or why not?

*Your answer:*
no because correlation shows a realtionship not cause and affect.

## Bonus

In Q2 you ran a chi-square test. Look at its **expected counts** (not the observed counts). Do they all meet the assumption you learned in the chi-square module? What would you tell the company about how much to trust that result?

In [ ]:
# hint: chi2_contingency returns the expected table as its 4th value
# check: are all expected counts >= 5?


*Your answer:*
no. 2 expected counts were 4.5 which is lower than 5. i would tell the company to not trust it fully.

### Push your code to GitHub